In [1]:
import pandas as pd
import re

import torch

In [2]:
#extracting datasets
df_cnbc = pd.read_csv('Dataset/cnbc_headlines.csv')
df_guardian = pd.read_csv('Dataset/guardian_headlines.csv')
df_reuters = pd.read_csv('Dataset/reuters_headlines.csv')

#considering only headlines
headline_cnbc = list(df_cnbc['Headlines'])
headline_guardian = list(df_guardian['Headlines'])
headline_reuters = list(df_reuters['Headlines'])

#list of headlines
headline_list = headline_cnbc + headline_guardian + headline_reuters

print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#cleaning dataset

#filtering empty rows
headline_list = [h for h in headline_list if str(h) != 'nan']
print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#removing spaces in headline
def clean_headline(h):
    h = h.strip() #remove the trailing spaces from either end
    h = re.sub(r'\s+', ' ', h) #replace bigger spaces with a single space
    return h

headline_list = [clean_headline(h) for h in headline_list]

No.of headlines in list:  53650
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", nan, "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally']
No.of headlines in list:  53370
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally', "Wall Street delivered the 'kind of pullback I've been waiting for,' Jim Cramer says"]


In [3]:
context_x = [] #list of all context windows
targets_y = [] #list of all targets

for headline in headline_list:
    context_window = ['<S>','<S>','<S>']
    chs = list(headline) + ['<E>']
    for char in chs:
        context_x.append(context_window)
        targets_y.append(char)
        context_window = context_window[1:] + [char]
        
#firstly, loop through every headline in the list of headlines
#then, append each context window mapping to each target character, and slide the window one element to the right, to create the next context window and repeat the same

print(f"Total training pairs: {len(context_x)}")
print(f"Example context: {context_x[0]}, target: {targets_y[0]}")
print(f"Example context: {context_x[10]}, target: {targets_y[10]}")

Total training pairs: 3594337
Example context: ['<S>', '<S>', '<S>'], target: J
Example context: ['m', 'e', 'r'], target: :


In [4]:
#vocabulary set
unique_words = set()

for headline in headline_list:
    for letter in list(headline):
        unique_words.add(letter)
    
unique_words = ['<S>', '<E>'] + sorted(unique_words)
print("Vocabulary size: ",len(unique_words))

#number to word matrix
num2letter_dict = {i:letter for i, letter in enumerate(unique_words)}

#word to number matrix
letter2num_dict = {letter:i for i, letter in enumerate(unique_words)}

Vocabulary size:  110


In [6]:
#converting x(context window) into tensors
context_indices = [[letter2num_dict[ch] for ch in context] for context in context_x ]
X = torch.tensor(context_indices)

#converting y(targets) into tensors
target_indices = [letter2num_dict[ch] for ch in targets_y]
Y = torch.tensor(target_indices)

print("Context window tensor size: ", X.shape)
print("Target tensor size: ", Y.shape)
print(X[:3])
print(Y[:3])

Context window tensor size:  torch.Size([3594337, 3])
Target tensor size:  torch.Size([3594337])
tensor([[ 0,  0,  0],
        [ 0,  0, 40],
        [ 0, 40, 66]])
tensor([40, 66, 70])


In [7]:
#creating an embedding table
C = torch.randn([110,10]) #creates a matrix of 110x10 with random values from a normal distribution
emb = C[X] #a batch lookup to match the embedding matrix with the context window matrix
print(emb.shape)

torch.Size([3594337, 3, 10])


In [9]:
#flatten emb to form a hidden layer
emb_flat = emb.view(emb.shape[0], -1)
print(emb_flat.shape)

torch.Size([3594337, 30])
